# RetailPulse analytics workflow (Days 4–11)
This notebook is a reproducible walkthrough of the feature, RFM, time-aware baseline forecast, and inventory review modules. It expects Day 2 processed CSVs in data/processed. It does not edit data/raw. The notebook is committed without execution outputs; run it locally to generate actual results.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))
from features.build_features import build_features
from segmentation.customer_segmentation import build_rfm
from forecasting.forecast_sales import evaluate_baseline
from inventory.reorder_analysis import build_reorder_review
import pandas as pd
DATA = ROOT / 'data' / 'processed'


## Day 4: feature engineering
Transaction calendar features and product sales/inventory summaries are derived from processed data.

In [ ]:
features = build_features(DATA)
{name: frame.shape for name, frame in features.items()}


## Days 5–6: customer RFM
Recency uses the day after the most recent transaction in the supplied file. Scores and segments are relative to this snapshot.

In [ ]:
sales = features['sales']
rfm = build_rfm(sales)
rfm.groupby('segment').agg(customers=('customer_id','nunique'), revenue=('monetary','sum')).sort_values('customers', ascending=False)


## Days 7–9: time-aware sales baseline
The final three calendar months form a chronological holdout. This naive baseline is a learning benchmark, not a production forecast.

In [ ]:
train, holdout, metrics = evaluate_baseline(sales)
metrics, holdout


## Days 10–11: inventory review
The review uses the latest observed snapshot per product and average sales velocity over the available transaction window. Purchase orders and stockout censoring are not available.

In [ ]:
inventory = pd.read_csv(DATA / 'inventory_clean.csv')
products = pd.read_csv(DATA / 'products_clean.csv')
review = build_reorder_review(sales, inventory, products)
review.loc[review.review_flag].head(20)


## Interpretation limits
The source has a small, irregular history and the latest month may be partial. Churn prediction is not included because no defensible outcome label/window is supplied. See reports/roadmap_completion.md.